## Client node to handle robot goals

This notebook node lets the user send/cancel goals, monitor robot state, and visualize progress.

## Imports and Global Variables

In [1]:

import rospy
from assignment2_rt_part1.msg import RobotState, Target
from sensor_msgs.msg import LaserScan
from nav_msgs.msg import Odometry
from assignment_2_2024.msg import PlanningAction, PlanningGoal
import actionlib
import threading
import time
import ipywidgets as widgets
from IPython.display import display, clear_output
%matplotlib widget
import matplotlib.pyplot as plt 
import tf
from tf.transformations import quaternion_matrix 
import numpy as np
from matplotlib.animation import FuncAnimation



# Global variables
robot_state = RobotState()
client = None
min_obstacle_distance = None
stop_event = threading.Event()
target_pub = None
ros_initialized = False
goal_viz = None

# UI widgets

# Widget for visualizing the state of the robot
robot_state_display = widgets.Label()
# Widget for visualizing the closest obstacle
obstacle_distance_display = widgets.Label()
# Widget input
x_input = widgets.FloatText(description='Target X:')
y_input = widgets.FloatText(description='Target Y:')
send_button = widgets.Button(description='Send Goal', button_style='success')
cancel_button = widgets.Button(description='Cancel Goal', button_style='danger')
output_area = widgets.Output()

## Callback functions


In [2]:
# Callback function to receive odometry data and update position and velocity
def odom_callback(msg):
    global robot_state
    # Get position and velocity from odometry data
    robot_state.x = msg.pose.pose.position.x
    robot_state.y = msg.pose.pose.position.y
    robot_state.vel_x = msg.twist.twist.linear.x
    robot_state.vel_z = msg.twist.twist.angular.z
    state_pub.publish(robot_state)

# Callabck function to find the closest obstacle
def scan_callback(msg):
    global min_obstacle_distance
    # Filter valid distances
    valid_ranges = [r for r in msg.ranges if msg.range_min < r < msg.range_max]
    if valid_ranges:
        min_obstacle_distance = min(valid_ranges)
    else:
        min_obstacle_distance = None
        
# Callback function to check if the goal is reached
def done_callback(state, result):
    with output_area:
        if state == actionlib.GoalStatus.SUCCEEDED:
            print("Goal reached!")
            goal_viz.update_counts(success=True)
        else:
            print("Goal not reached or canceled.")
            goal_viz.update_counts(success=False)



## Display related functions

In [3]:
# Function to update the visualization ot the position and the velocity of the robot in real time
def update_display():
    while not stop_event.is_set():
        pos = f"Position: (x: {robot_state.x:.2f}, y: {robot_state.y:.2f})"
        vel = f"Velocity: lin {robot_state.vel_x:.2f}, ang {robot_state.vel_z:.2f}"
        robot_state_display.value = f"{pos} | {vel}"

        if min_obstacle_distance is not None:
            obstacle_distance_display.value = f"Closest obstacle: {min_obstacle_distance:.2f} m"
        else:
            obstacle_distance_display.value = "Closest obstacle: N/A"
        
        time.sleep(0.5)

In [4]:
# Function to send a goal to the action server
def send_goal_clicked(b):
    with output_area:
        clear_output()  # Clear the output area for status updates
        
        # Cancel only if the previous goal is still active or pending
        if client.get_state() in [actionlib.GoalStatus.ACTIVE, actionlib.GoalStatus.PENDING]:
            client.cancel_goal()
            rospy.sleep(0.5)
        
        goal = PlanningGoal()
        goal.target_pose.pose.position.x = x_input.value
        goal.target_pose.pose.position.y = y_input.value
        client.send_goal(goal, done_cb=done_callback)

        target_msg = Target(x=x_input.value, y=y_input.value)
        target_pub.publish(target_msg)

        print(f" New goal sent: ({x_input.value}, {y_input.value})")


# Function to cancel a sent goal
def cancel_goal_clicked(b):
    with output_area:
        clear_output()
        if client.get_state() in [actionlib.GoalStatus.ACTIVE, actionlib.GoalStatus.PENDING]:
            client.cancel_goal()
            print("Goal cancelled.")
        else:
            print("No active goal to cancel.")



## ROS setup

In [5]:
def setup_ros():
    global client, target_pub, state_pub, ros_initialized
    
    if ros_initialized:
        return
    ros_initialized = True

    rospy.init_node('jupyter_interface_node', anonymous=True)
    
    # Subscribers
    rospy.Subscriber('/odom', Odometry, odom_callback)
    rospy.Subscriber('/scan', LaserScan, scan_callback)

    # Publisher
    target_pub = rospy.Publisher('target_topic', Target, queue_size=10)
    state_pub  = rospy.Publisher('/robot_state', RobotState, queue_size=10)

    client = actionlib.SimpleActionClient('/reaching_goal', PlanningAction)
    rospy.loginfo("Waiting for action server to start...")
    client.wait_for_server()
    rospy.loginfo("Action server started.")

    # Thread to update the state
    t = threading.Thread(target=update_display)
    t.daemon = True
    t.start()

setup_ros()

[INFO] [1747060920.983603, 4362.773000]: Waiting for action server to start...
[INFO] [1747060921.075778, 4362.855000]: Action server started.


## Realtime robot path plot

In [6]:
class Visualiser:
    def __init__(self):
        plt.ioff()
        self.fig, self.ax = plt.subplots(figsize=(3.7,2))
        plt.ion()
        self.ln, = self.ax.plot([], [], color='orange', linewidth=3)
        self.x_data, self.y_data = [] , []
        self.ax.set_title("Robot path plot")

    def plot_init(self):
        self.ax.set_xlim(-10, 10)
        self.ax.set_ylim(-10, 10)
        return self.ln,

    def odom_callback(self, msg):
        self.y_data.append(msg.pose.pose.position.y)
        self.x_data.append(msg.pose.pose.position.x)

    def update_plot(self, frame):
        self.ln.set_data(self.x_data, self.y_data)
        return self.ln,



## Goal Outcome Histogram

In [7]:
class GoalOutcomeVisualizer:
    def __init__(self):
        plt.ioff()
        self.reached = 0
        self.not_reached = 0

        self.fig, self.ax = plt.subplots(figsize=(3.7,2))
        self.bars = None
        plt.ion()
        self.categories = ['Reached', 'Not Reached']
        self.colors = ['green', 'red']

        self.plot_init()

    def plot_init(self):
        self.ax.set_title("Goal Outcomes")
        self.ax.set_ylabel("Count")
        self.bars = self.ax.bar(self.categories, [self.reached, self.not_reached], color=self.colors)
        self.ax.set_ylim(0, 10)  # Adjustable limit
        self.fig.canvas.draw_idle()

    def update_counts(self, success):
        if success:
            self.reached += 1
        else:
            self.not_reached += 1
        self.update_plot()

    def update_plot(self):
        counts = [self.reached, self.not_reached]
        for bar, height in zip(self.bars, counts):
            bar.set_height(height)

        # Adjust y-axis limit if needed
        self.ax.set_ylim(0, max(10, self.reached + self.not_reached + 1))
        self.fig.canvas.draw_idle()



## UI layout

In [8]:
# Function to update the plots dynamically inside Output widget
def update_plots(frame):
    # Update robot path plot
    vis.update_plot(frame)
    # Update goal outcome plot
    goal_viz.update_plot()  
    return vis.ln, goal_viz.bars

send_button.on_click(send_goal_clicked)
cancel_button.on_click(cancel_goal_clicked)

# Control UI
control_ui = widgets.VBox([
    robot_state_display,
    obstacle_distance_display,
    widgets.HBox([x_input, y_input]),
    widgets.HBox([send_button, cancel_button]),
    output_area
], layout=widgets.Layout(width='700px'))

# Visualizer for the goals
goal_viz = GoalOutcomeVisualizer()

# Initialize the Visualiser for the robot path
vis = Visualiser()
sub = rospy.Subscriber('/odom', Odometry, vis.odom_callback)
ani = FuncAnimation(vis.fig, update_plots, init_func=vis.plot_init, interval=1000, blit=False) # Initialize the animation

# Plots box
plots_box = widgets.HBox(
    [vis.fig.canvas, goal_viz.fig.canvas],
    layout=widgets.Layout(
        border='1px solid lightgray',
        padding='5px',
        margin='5px 0',
        align_items='stretch'
    )
)

# Create the full UI layout with control_ui at the top and plots_box below
full_ui = widgets.VBox([
    control_ui,  
    plots_box    
], layout=widgets.Layout(
    width='900px',
    margin='10px 0'
))



## Main

In [9]:

display(full_ui)
